In [ ]:
#| default_exp game/globals

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
#from fasthtml.jupyter import get_host
from fastlite import *
import fasthtml.components as fc
import pandas as pd
import httpx
import threading

In [ ]:
#| export
from HexMagic.core import Terrain, DrainageBasins
from HexMagic.game.data import GameBoard

In [ ]:
#| export
from monsterui.all import *


The `apsw.ThreadingViolationError` happens because APSW (which fastlite uses under the hood) checks that a connection is only used from the thread that created it. Since Starlette/uvicorn runs your route handlers in a **thread pool**, multiple request threads can hit `globalStore.db` concurrently.

This isn't really an async/await issue — it's a **shared connection across threads** issue. Here are two clean fixes:

**Option 1: Threading Lock (simplest)**

Serialize all DB access with a lock:

```python
import threading
db_lock = threading.Lock()

def ensure_user(session):
    with db_lock:
        uid = session.get('uid')
        row = globalStore.db.execute("SELECT id FROM user WHERE id = ?", [uid]).fetchone()
        # ... rest of db work
```

Wrap every function that touches the DB in `with db_lock:`. Simple, and perfectly fine for a game with moderate concurrency.

**Option 2: Thread-local connections**

Give each thread its own connection:

```python
import threading
_local = threading.local()

def get_db():
    if not hasattr(_local, 'db'):
        _local.db = database('hex.db')  # each thread gets its own connection
    return _local.db
```

Then use `get_db()` instead of `globalStore.db` in your route handlers. This allows true concurrent reads (especially with WAL mode).

**Which to pick?**

- **Lock** is easier — just wrap existing code, no structural changes. Fine for low-to-moderate traffic.
- **Thread-local** is better if you want concurrent reads, but you need to be careful about write conflicts.

For either approach, also consider enabling WAL mode on your database for better concurrent read performance:

```python
globalStore.db.execute("PRAGMA journal_mode=WAL")
```

I'd start with the threading lock since it's a one-line change per function and your current code structure stays the same.

In [ ]:
#| export
def appRoutes():
    global app, rt, hexserver
    #if 'hexserver' not in dir() or hexserver is None:
    if 'hexserver' not in globals() or hexserver is None:

        app = FastHTML(hdrs=Theme.violet.headers())
        rt = app.route
        hexserver = JupyUvi(app)
    return app, rt, hexserver


In [ ]:
#| export
app, rt,  hexserver = appRoutes()

So I want app rt and hexserver only intiate once, but this will be shared among notebooks. how do I do this patter

In [ ]:
#| export
def webMe(*c): return HTMX(*c, host='', app=app)

In [ ]:
@rt
def hello(name: str): return P(f"Hello {name}!")

webMe(Div(
    H3("Say Hello"),
    Form(
        Input(placeholder="Your name...", id='name'),
        Button("Send"),
        hx_get=hello, hx_target="#result"
    ),
    Div(id='result'),
))

In [ ]:
#| export
from HexMagic.game.data import ActiveGame, GameStorage, TerrainTemplate, Settlement, Piece

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User

In [ ]:
#| export
import logging

logging.basicConfig(
    filename='base.text',
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

logging.info("getting Started")


In [ ]:
read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

## Helpers

## ActiveGame

## Database

In [ ]:
#| export
globalStore = GameStorage("gameData/WebDebug.db")

In [ ]:
#| export
# Helper to ensure we have a user row
def ensure_user(session) -> int:
    if 'userid' not in session:
        session['userid'] = random.randint(0, 1_000_000)
    uid = session['userid']
    row = globalStore.db.execute("SELECT id FROM user WHERE id = ?", [uid]).fetchone()
    if not row:
        from datetime import datetime
        now = int(datetime.now().timestamp())
        globalStore.users.insert({
            'id': uid, 'username': f'player_{uid}', 'email': '', 'password': '',
            'created': now, 'sessionID': str(uid), 'activeWorld': 0
        })
    return uid



def new_game_page():
    templates = TerrainTemplate.maps  # {'bayArea': 'bayArea_map', ...}
    form = Form(
        Div(
            Label("Map Template", cls="label"),
            Select(
                *[Option(name, value=name) for name in sorted(templates.keys())],
                name="template_name", cls="select select-bordered w-full"
            ),
            cls="form-control"
        ),
        Div(
            Label("Kingdoms", cls="label"),
            Input(type="number", name="kingdoms", value="5", min="1", max="10",
                  cls="input input-bordered w-full"),
            cls="form-control"
        ),
        Div(
            Label("Lakes", cls="label"),
            Input(type="number", name="lakes", value="1", min="0", max="5",
                  cls="input input-bordered w-full"),
            cls="form-control"
        ),
        Div(
            Label("Hex Radius", cls="label"),
            Input(type="number", name="radius", value="25", min="10", max="40",
                  cls="input input-bordered w-full"),
            cls="form-control"
        ),
        Button("Create World", type="submit", cls="btn btn-primary mt-4"),
        action="/create_world", method="post",
        cls="card bg-base-200 shadow-lg p-6 space-y-4 max-w-md mx-auto"
    )
    return Titled("New Game",
        Div(
            H3("Choose Your World", cls="text-2xl font-bold text-center mb-6"),
            form,
            cls="flex flex-col items-center p-12"
        )
    )




In [ ]:
#| export
@patch
def create_game(self: GameStorage, user_id, template_name="bayArea",
                kingdoms=5, lakes=1, radius=10) -> ActiveGame:
    logging.info(f"create_game: user={user_id} template={template_name}")
    tt = TerrainTemplate()
    terrain = getattr(tt, template_name)()
    
    terrain.carve_to_ocean(num_lakes=lakes)
    terrain.hexGrid.adjustRadius(radius)

    # GameBoard now creates cover + basins internally
    board = GameBoard(terrain, top_n=kingdoms)
    board.expand_kingdoms(max_rounds=50)
    
    # Save using the cover that GameBoard created
    board.cover.db = self
    board.cover.save(name=template_name)
    board.save(self, world_id=board.cover.ident)

    self.db.execute("UPDATE user SET activeWorld = ? WHERE id = ?",
                    [board.cover.ident, user_id])
    
    return ActiveGame(board=board, cover=board.cover, world_id=board.cover.ident)


In [ ]:
read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

## Common Routes

@rt
def create_world(session, template_name: str, kingdoms: int = 5,
                 lakes: int = 1, radius: int = 10):
    uid = ensure_user(session)
    globalStore.create_game(uid, template_name=template_name,
                            kingdoms=kingdoms, lakes=lakes, radius=radius)
    logging.info(f"create world redirecting")
    return RedirectResponse('/game', status_code=303)


In [ ]:
#| export
@rt
def create_world(session, template_name: str, kingdoms: int = 5,
                 lakes: int = 1, radius: int = 10):
    uid = ensure_user(session)
    logging.info(f"create_world: session keys={list(session.keys())}")
    invalidate_cache(uid)  # <-- clear old game
    globalStore.create_game(uid, template_name=template_name,
                            kingdoms=kingdoms, lakes=lakes, radius=radius)
    return RedirectResponse('/game', status_code=303)

In [ ]:
#| export
_game_cache = {}
_cache_lock = threading.Lock()

def invalidate_cache(user_id):
    """Call this when creating a new game or the board changes."""
    with _cache_lock:
        _game_cache.pop(user_id, None)


## Overlays

In [ ]:
#| export
# ── Overlay session helpers ──────────────────────────────────

DEFAULT_OVERLAYS = {'settlements', 'names', 'elevation', 'rivers'}

OVERLAY_CONFIG = {
    'cream':       {'label': '🏔️ Terrain',       'fill': '#F5DEB3', 'stroke': '#8B7355'},
    'rivers':      {'label': '🌊 Rivers',        'fill': '#4169E1', 'stroke': '#1E3A6E'},
    'names':       {'label': '📝 Names',         'fill': '#4A90D9', 'stroke': '#2C5F8A'},
    'settlements': {'label': '🏰 Settlements',   'fill': '#DAA520', 'stroke': '#8B6914'},
    'temperature': {'label': '🌡️ Temperature',   'fill': '#E8734A', 'stroke': '#A34520'},
    'climate':     {'label': '🌿 Climate',       'fill': '#7A9B76', 'stroke': '#4A6B46'},
    'watersheds':  {'label': '💧 Watersheds',    'fill': '#6B8EC4', 'stroke': '#3A5E94'},
    'flow':        {'label': '🧭 Flow',          'fill': '#555555', 'stroke': '#333333'},
    'elevation':   {'label': '⛰️ Elevation',     'fill': '#9D8B73', 'stroke': '#6D5B43'},
}


def get_overlays(session) -> set:
    raw = session.get('overlays')
    if raw is None: return set(DEFAULT_OVERLAYS)
    return set(raw)

def set_overlays(session, overlays: set):
    session['overlays'] = list(overlays)


In [ ]:
#| export
@patch
def apply_overlays(zoomed: Terrain, result: ZoomResult, board: GameBoard,
                   c2f: dict, overlays: set, settle_attrs: dict = None,level:int=3):
    """Configure terrain builder layers based on active overlay set.
    
    Pure helper — no session, no redirects, no route logic.
    """
    builder = zoomed.hexGrid.builder

    # Always show borders
    builder.adjust("borders", board.countries_overlay(zoomed, c2f))

    if 'cream' in overlays:
        zoomed.terrainCream()

    if 'elevation' in overlays:
        builder.adjust("elevation", zoomed.elevation_borders())

    if 'temperature' in overlays:
        try:
            builder.adjust("temperature", zoomed.render_icon_temperature())
        except ValueError:
            pass  # fields not computed

    if 'climate' in overlays:
        try:
            builder.adjust("climate", zoomed.dottedClimate())
        except ValueError:
            pass

    if 'watersheds' in overlays and result.basins:
        builder.adjust("watersheds", result.basins.dotted_watershed_overlay())

    if 'rivers' in overlays and result.basins:
        builder.adjust("water", result.basins.draw_watersheds())

    if 'flow' in overlays:
        try:
            builder.adjust("flow", zoomed.flow_diagram())
        except Exception:
            pass

    if 'names' in overlays:
        builder.adjust("names", board.names_overlay(zoomed, c2f))

    if 'settlements' in overlays and settle_attrs:
        builder.adjust("settlements",
                       board.settlementTouchOverlay(zoomed, c2f, attrs=settle_attrs))


In [ ]:
#| export
@rt("/toggle_overlay/{id}")
def toggle_overlay(session, id: str, overlay: str):
    """Toggle a single overlay on/off and re-render the map + controls."""
    current = get_overlays(session)
    if overlay in current:
        current.discard(overlay)
    else:
        current.add(overlay)
    set_overlays(session, current)

    # Re-render both the map and the controls bar
    map_partial = settlement_map(session, id)
    controls_partial = Div(
        settlement_controls(session, id),
        id="map-controls",
        hx_swap_oob="true"
    )
    return map_partial, controls_partial


In [ ]:
#| export
# ── Controls route — HexLegend toggle bar ────────────────────

@rt("/overlay_controls/{id}")
def overlay_controls(session, id: str):
    """Bottom bar with toggleable overlay hex swatches."""
    active = get_overlays(session)

    hexes, values = [], []
    for key, cfg in OVERLAY_CONFIG.items():
        is_on = key in active
        fill   = cfg['fill']   if is_on else '#888'
        stroke = cfg['stroke'] if is_on else '#555'
        opacity = '1.0' if is_on else '0.35'

        style = StyleCSS(f"ov_{key}", fill=fill, stroke=stroke,
                         stroke_width=2, opacity=opacity)
        label = cfg['label'] + (' ✓' if is_on else '')
        h = Hex(radius=15, center=MapCord(0, 0), style=style, label=label)
        hexes.append(h)
        values.append(key)

    return HexLegend(
        hexes, name="overlay", values=values,
        hx_post=f"/toggle_overlay/{id}",
        hx_target="#map", hx_swap="outerHTML",
        direction="row", size=30, font_size=11,
        gap=4, item_gap=16,
    )


dummySession= {'userid': 667256 }
webMe(index(dummySession))

In [ ]:
!tail -10 base.text

In [ ]:
#| export
# Get all users
def showUsers():
    users_df = pd.DataFrame(globalStore.users())
    print("Users:")
    print(users_df)

In [ ]:
showUsers()

In [ ]:
dummySession= {'userid': 64801 }
#webMe(index(dummySession))

SVGBuilder.BUILDERHIDE = True

myRessult = globalStore.active_board(64801)
myBoard = myRessult.board
myTerrain = myBoard.terrain
show(myTerrain)

worldId = 3
k = next((k for k in myBoard.kingdoms if k.countryId == worldId), None)
if k:
    print(k.countryName)
#result = globalStore.kingdom_detail(myRessult.world_id, worldId, myRessult.cover)
#show(result.terrain)

k.settlements